# Reactor Overall Yield Prediction

A reproducible machine learning workflow for predicting **overall reactor yield** from operating conditions.

This notebook covers data loading, exploratory analysis, physics-informed feature engineering, cross-validation, model comparison, hyperparameter tuning, ensemble evaluation, final model selection, and test-set prediction.

## 1. Problem Overview

This is a **supervised regression** problem. The objective is to predict the continuous target `overall_yield` from five reactor operating conditions.

| Variable | Description | Role |
|---|---|---|
| `flow_rate_L_min` | Volumetric feed rate through the reactor | Feature |
| `concentration_mol_L` | Inlet reactant concentration | Feature |
| `inlet_temperature_K` | Feed temperature entering the reactor | Feature |
| `length_m` | Reactor tube length | Feature |
| `jacket_temperature_K` | Cooling/heating jacket temperature | Feature |
| `overall_yield` | Final reactor yield (%) | Target |

**Evaluation metric:** Root Mean Squared Error (RMSE).

Because the dataset is relatively small, the workflow emphasizes robust cross-validation and interpretable feature engineering rather than relying on training-set performance.


## 2. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, VotingRegressor, StackingRegressor,
)
from sklearn.model_selection import (
    KFold, RepeatedKFold, cross_val_score, RandomizedSearchCV, GridSearchCV,
)
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Optional gradient boosting libraries (used only if installed)
HAS_XGB = HAS_CAT = False
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    pass
try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except Exception:
    pass
print("xgboost:", HAS_XGB, "| catboost:", HAS_CAT)

## 3. Load the Datasets

In [ ]:
train = pd.read_csv("train_dataset.csv")
test  = pd.read_csv("test_dataset.csv")

TARGET = "overall_yield"
BASE_FEATURES = [
    "flow_rate_L_min", "concentration_mol_L", "inlet_temperature_K",
    "length_m", "jacket_temperature_K",
]

print("train shape:", train.shape, "| test shape:", test.shape)
display(train.head())
display(test.head())
print(train.dtypes)
display(train.describe().T)

## 4. Exploratory Data Analysis

The exploratory analysis focuses on data quality, target distribution, feature relationships, and potentially important operating regimes.

Key checks performed in this section include:

- Missing-value analysis
- Duplicate-row detection
- Descriptive statistics
- Target distribution
- Feature-to-target relationships
- Correlation analysis

The visualizations are used to understand the structure of the dataset and guide the subsequent feature-engineering strategy.


In [ ]:
print("Missing values per column:\n", train.isna().sum(), "\n")
print("Duplicate rows:", train.duplicated().sum())
print("Duplicate feature rows:", train.duplicated(subset=BASE_FEATURES).sum(), "\n")

summary = train.agg(["min", "max", "mean", "median", "std"]).T
display(summary)
print("Rows with ~zero yield:", int((train[TARGET] <= 1e-3).sum()), "of", len(train))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(train[TARGET], bins=25, color="#2a9d8f", edgecolor="black")
ax[0].set_title("Target distribution: overall_yield"); ax[0].set_xlabel("overall_yield")
train[BASE_FEATURES].plot(kind="box", subplots=False, ax=ax[1], rot=45)
ax[1].set_yscale("log"); ax[1].set_title("Feature boxplots (log scale)")
plt.tight_layout(); plt.show()

train[BASE_FEATURES].hist(figsize=(12, 6), bins=20, color="#264653")
plt.suptitle("Feature distributions"); plt.tight_layout(); plt.show()

In [ ]:
corr = train[BASE_FEATURES + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im); plt.title("Correlation matrix"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6), sharey=True)
for ax, f in zip(axes, BASE_FEATURES):
    ax.scatter(train[f], train[TARGET], s=14, alpha=0.7, color="#e76f51")
    ax.set_xlabel(f)
axes[0].set_ylabel("overall_yield")
plt.suptitle("Every feature vs overall_yield")
plt.tight_layout(); plt.show()

### Interpreting the Feature Relationships

The exploratory results suggest several physically meaningful relationships:

- **Jacket temperature vs. yield:** jacket temperature is one of the strongest drivers of the observed yield behavior.
- **Inlet temperature vs. yield:** the interaction between inlet and jacket temperatures appears important, making temperature differences useful features.
- **Flow rate and reactor length vs. yield:** these variables influence residence time, motivating the use of `length / flow_rate`.
- **Concentration vs. yield:** concentration contributes to the prediction and may interact with temperature-related effects.

These observations motivate a compact set of physics-informed features rather than a large, highly expanded feature space.


## 5. Physics-Informed Feature Engineering

A compact set of interpretable features is generated from the original reactor operating conditions.

The objective is to expose physically meaningful relationships such as:

- Residence time
- Temperature differences and ratios
- Concentration-flow interactions
- Other directly derived operating-condition relationships

The feature set is deliberately kept small to reduce unnecessary complexity and the risk of overfitting on a limited dataset. All divisions are protected with a small epsilon value.


In [ ]:
EPS = 1e-6

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    flow = d["flow_rate_L_min"]; conc = d["concentration_mol_L"]
    tin  = d["inlet_temperature_K"]; ln = d["length_m"]; tj = d["jacket_temperature_K"]

    d["delta_T"]             = tj - tin                    # driving force for heat exchange
    d["temp_ratio"]          = tj / (tin + EPS)
    d["residence_time"]      = ln / (flow + EPS)           # contact time proxy
    d["inv_flow_rate"]       = 1.0 / (flow + EPS)
    d["conc_x_inlet_T"]      = conc * tin / 1000.0         # rate ~ k(T) * C
    d["conc_x_jacket_T"]     = conc * tj / 1000.0
    d["length_x_flow"]       = ln * flow / 100.0           # throughput proxy
    d["jacket_T_per_flow"]   = tj / (flow + EPS)           # heat load per unit flow
    d["log_residence_time"]  = np.log1p(d["residence_time"])
    d["delta_T_x_residence"] = d["delta_T"] * d["residence_time"] / 10.0
    return d.replace([np.inf, -np.inf], np.nan)

FEATURES = ["flow_rate_L_min","concentration_mol_L","inlet_temperature_K","length_m","jacket_temperature_K","delta_T","temp_ratio","residence_time","inv_flow_rate","conc_x_inlet_T","conc_x_jacket_T","length_x_flow","jacket_T_per_flow","log_residence_time","delta_T_x_residence"]

train_fe = add_features(train)
test_fe  = add_features(test)
X = train_fe[FEATURES].values
y = train_fe[TARGET].values
X_test = test_fe[FEATURES].values
print("X:", X.shape, "| X_test:", X_test.shape)

display(train_fe[FEATURES + [TARGET]].corr()[[TARGET]].sort_values(TARGET, key=abs, ascending=False))

Feature engineering consists only of **stateless, row-wise arithmetic** and does not learn statistics from the dataset.

Any preprocessing that learns from data, such as imputation and scaling, is handled inside scikit-learn `Pipeline` objects. This keeps preprocessing isolated within each cross-validation fold and prevents information leakage.


## 6. Train-Validation Strategy

Model performance is evaluated using repeated cross-validation rather than relying on a single train-validation split.

The primary evaluation uses **RepeatedKFold with 5 splits and 3 repeats**, while standard 5-fold and 10-fold cross-validation are also used as reference checks.

The evaluation metric is `neg_root_mean_squared_error`, and a fixed random seed of `42` is used for reproducibility.


In [ ]:
cv_repeated = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
cv_5  = KFold(n_splits=5,  shuffle=True, random_state=RANDOM_STATE)
cv_10 = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

def cv_rmse(model, cv=cv_repeated, X=X, y=y):
    s = cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    return -s.mean(), s.std()

def make_pipe(estimator, scale=False):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)

## 7. Model Comparison

Several regression model families are evaluated, including:

- Linear regression with regularization
- Instance-based regression
- Tree-based bagging methods
- Gradient boosting methods
- Kernel-based methods

The models are evaluated under the same cross-validation framework so that their performance can be compared consistently.

The goal is to identify models capable of capturing nonlinear relationships and sharp operating-regime boundaries while avoiding unnecessary model complexity.


In [ ]:
candidates = {
    "Ridge (alpha=1)":        make_pipe(Ridge(alpha=1.0, random_state=RANDOM_STATE), scale=True),
    "Ridge (alpha=10)":       make_pipe(Ridge(alpha=10.0, random_state=RANDOM_STATE), scale=True),
    "SVR (RBF, scaled)":      make_pipe(SVR(C=100, epsilon=1.0, gamma="scale"), scale=True),
    "RandomForest":           make_pipe(RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)),
    "RandomForest (reg.)":    make_pipe(RandomForestRegressor(n_estimators=500, max_depth=6, min_samples_leaf=3,
                                                              max_features=0.5, random_state=RANDOM_STATE, n_jobs=-1)),
    "ExtraTrees":             make_pipe(ExtraTreesRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)),
    "GradientBoosting":       make_pipe(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                                  max_depth=3, subsample=0.9, random_state=RANDOM_STATE)),
    "HistGradientBoosting":   make_pipe(HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08,
                                                                      max_depth=4, random_state=RANDOM_STATE)),
}
if HAS_XGB:
    candidates["XGBoost"] = make_pipe(XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=4,
                                                   subsample=0.9, colsample_bytree=0.9,
                                                   random_state=RANDOM_STATE, n_jobs=-1))
if HAS_CAT:
    candidates["CatBoost"] = make_pipe(CatBoostRegressor(iterations=800, learning_rate=0.05, depth=5,
                                                         random_seed=RANDOM_STATE, verbose=0))

rows = []
for name, model in candidates.items():
    m, s = cv_rmse(model)
    rows.append({"Model": name, "CV RMSE Mean": round(m, 4), "CV RMSE Std": round(s, 4)})

comparison = pd.DataFrame(rows).sort_values("CV RMSE Mean").reset_index(drop=True)
display(comparison)

In [ ]:
# Sanity check: does the CV scheme itself change the ranking?
best_name = comparison.iloc[0]["Model"]
for label, cv in [("5-fold", cv_5), ("10-fold", cv_10), ("Repeated 5x3", cv_repeated)]:
    m, s = cv_rmse(candidates[best_name], cv=cv)
    print(f"{best_name:25s} {label:14s} RMSE = {m:.4f} +/- {s:.4f}")

## 8. Hyperparameter Tuning

The strongest candidate model families are further optimized using a focused `RandomizedSearchCV` search space.

The same cross-validation strategy and RMSE-based scoring are retained during tuning. Preprocessing remains inside the model pipelines so that learned transformations are refit independently for each fold.


In [ ]:
search_spaces = {
    "GradientBoosting": (
        make_pipe(GradientBoostingRegressor(random_state=RANDOM_STATE)),
        {
            "model__n_estimators": [200, 300, 500, 800],
            "model__learning_rate": [0.02, 0.03, 0.05, 0.08],
            "model__max_depth": [2, 3, 4],
            "model__subsample": [0.7, 0.8, 0.9, 1.0],
            "model__min_samples_leaf": [1, 2, 3],
        },
    ),
    "ExtraTrees": (
        make_pipe(ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        {
            "model__n_estimators": [300, 500, 800],
            "model__max_depth": [None, 8, 12, 20],
            "model__max_features": [0.5, 0.6, 0.8, 1.0],
            "model__min_samples_leaf": [1, 2, 3],
        },
    ),
    "RandomForest": (
        make_pipe(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        {
            "model__n_estimators": [300, 500, 800],
            "model__max_depth": [None, 6, 8, 12],
            "model__max_features": [0.4, 0.5, 0.6, 0.8],
            "model__min_samples_leaf": [1, 2, 3],
        },
    ),
}

tuned = {}
for name, (pipe, space) in search_spaces.items():
    search = RandomizedSearchCV(pipe, space, n_iter=25, cv=cv_repeated,
                                scoring="neg_root_mean_squared_error",
                                random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X, y)
    tuned[f"{name} (tuned)"] = search.best_estimator_
    print(f"{name}: RMSE {-search.best_score_:.4f} | {search.best_params_}")

## 9. Ensemble Evaluation

After evaluating individual models, ensemble approaches are tested using the strongest candidates.

An ensemble is retained only when its cross-validated performance provides a measurable improvement over the individual models under the same evaluation procedure.


In [ ]:
pool = {**candidates, **tuned}
ranked = sorted(pool.items(), key=lambda kv: cv_rmse(kv[1])[0])[:3]
top = [(n, m) for n, m in ranked]
print("Top 3:", [n for n, _ in top])

ens = {
    "Voting (top 2)": VotingRegressor([(n.replace(" ", "_"), m) for n, m in top[:2]]),
    "Voting (top 3)": VotingRegressor([(n.replace(" ", "_"), m) for n, m in top]),
    "Weighted blend (3:2:1)": VotingRegressor([(n.replace(" ", "_"), m) for n, m in top], weights=[3, 2, 1]),
    "Stacking (Ridge meta)": StackingRegressor(
        [(n.replace(" ", "_"), m) for n, m in top],
        final_estimator=Ridge(alpha=1.0), cv=cv_5, n_jobs=-1),
}

ens_rows = []
for name, model in ens.items():
    m, s = cv_rmse(model)
    ens_rows.append({"Model": name, "CV RMSE Mean": round(m, 4), "CV RMSE Std": round(s, 4)})
display(pd.DataFrame(ens_rows).sort_values("CV RMSE Mean"))

## 10. Final Model Selection

The final model is selected automatically using the lowest repeated cross-validation RMSE across the evaluated individual models, tuned models, and ensembles.

This makes model selection metric-driven and reproducible rather than based on model complexity or assumptions about which algorithm should perform best.


In [ ]:
all_models = {**pool, **ens}
scores = {name: cv_rmse(model) for name, model in all_models.items()}
leaderboard = (pd.DataFrame([{"Model": n, "CV RMSE Mean": m, "CV RMSE Std": s}
                             for n, (m, s) in scores.items()])
               .sort_values("CV RMSE Mean").reset_index(drop=True))
display(leaderboard)

best_name = leaderboard.iloc[0]["Model"]
best_model = all_models[best_name]
best_mean, best_std = scores[best_name]
print("Selected model      :", best_name)
print("Best CV RMSE        : %.4f" % best_mean)
print("CV RMSE std         : %.4f" % best_std)
print("Hyperparameters     :", getattr(best_model, "get_params", lambda: {})())

### Reference Model-Selection Results

The following results are retained as a reference from the existing notebook run.

| Model | CV RMSE Mean | CV RMSE Std |
|---|---:|---:|
| Voting ensemble (top 2, equal weights) | 19.0555 | 4.6597 |
| Weighted blend (top 3, 3:2:1) | 19.0908 | 4.2955 |
| Voting ensemble (top 3, equal weights) | 19.1213 | 4.2970 |
| Extra Trees | 19.3760 | 3.6179 |
| GBM tuned (n=300, lr=0.06, depth=4, sub=0.7) | 19.5518 | 5.7979 |
| ExtraTrees tuned (n=250, depth=12, mf=0.8) | 19.5877 | 3.6379 |
| HistGradientBoosting-style GBM | 20.1116 | 5.5070 |
| ExtraTrees tuned (n=300, depth=20, mf=0.5) | 20.2956 | 3.7016 |
| GBM tuned (n=300, lr=0.04, depth=3, sub=0.8) | 20.3638 | 5.4213 |
| Gradient Boosting | 20.5250 | 6.1584 |
| GBM tuned (n=200, lr=0.06, depth=3, sub=0.8) | 20.6059 | 5.3957 |
| ExtraTrees tuned (n=250, depth=8, mf=0.6) | 20.6835 | 3.4286 |
| Random Forest | 21.4306 | 4.4683 |
| Random Forest (regularized) | 21.7901 | 4.5949 |
| Ridge (alpha=1.0) | 28.1975 | 1.7228 |
| Ridge (alpha=10.0) | 29.0465 | 1.7322 |
| KNN (k=5, distance-weighted) | 31.4304 | 3.2010 |

**Selected model:** Voting ensemble (top 2, equal weights)

**Reference CV RMSE:** `19.0555 ± 4.6597`


## 11. Train Final Model on All Data

In [ ]:
best_model.fit(X, y)
print("Refit on the full training set:", X.shape[0], "rows")

## 12. Generate Test Predictions

The selected model is used to generate predictions for the test dataset.

The original test-row order is preserved, and predictions are constrained to the physically valid yield range of **0 to 100**.


In [ ]:
preds = best_model.predict(X_test)
preds = np.clip(preds, 0.0, 100.0).astype(float).round(3)

assert len(preds) == len(test), "Prediction count must match test rows"
print(preds[:10])

## 13. Generate Submission CSV

In [ ]:
submission = pd.DataFrame({"overall_yield": preds})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
display(submission.head())

## 14. Summary

This notebook implements an end-to-end reactor yield prediction workflow:

- Exploratory analysis is used to understand data quality and feature relationships.
- Physics-informed, row-wise feature engineering creates a compact and interpretable feature set.
- Cross-validation provides a more reliable estimate of model performance on the small dataset.
- Multiple regression families are compared using the same RMSE-based evaluation.
- Focused hyperparameter tuning is applied to the strongest model families.
- Ensemble models are evaluated against individual models under the same validation procedure.
- The final model is selected using cross-validated RMSE.
- The selected model is retrained on the complete training dataset and used to generate `submission.csv`.

The resulting CSV contains a single `overall_yield` column with predictions in the original test-row order.
